# Download, aggregate, and visualize filtered activations

This notebook selects completion records by prompt metadata, downloads their cached activations from GCS, aggregates cached token positions, projects the resulting feature vectors with PCA, and visualizes the projections.

The workflow is organized into six stages: setup, configuration, data selection and download, activation aggregation, PCA projection, and visualization.

## 1. Setup

### 1.1 Optional Colab bootstrap

Uncomment and run the next cell only when starting from a fresh Colab runtime. Local runs can skip it.

In [ ]:
# !git clone -b dev https://github.com/justinshenk/temporal-manifolds.git
# %cd temporal-manifolds
# !gcloud auth application-default login
# !mv -n .env.example .env

### 1.2 Imports and repository paths

Resolve the repository root and import the activation download, filtering, and aggregation utilities.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from temporal_manifolds.activations.extract_activations import load_selected_node_groups
from temporal_manifolds.utils.activation_aggregation import (
    aggregate_activation_file,
    nodes_for_classes,
)
from temporal_manifolds.utils.completion_filters import (
    download_activation_files,
    find_activation_paths,
)

## 2. Configure the workflow

Edit the next cell before running the download and analysis stages.

### Selection and download controls

- `PROMPT_FRAMING`, `OUTPUT_FORMAT`, and `TASK_METADATA_FILTERS` select completion records. Use `None` to disable a filter; every entry in `TASK_METADATA_FILTERS` must match.
- `MAX_FILES` caps the selected activation files, while `OVERWRITE` controls whether existing local files are replaced.

### Activation controls

- `NODE_CLASSES` unions selected-node classes; `None` includes every available class.
- `RESIDUAL_STREAM_LAYERS` uses `None` for every layer, a set of layer numbers for a subset, or `set()` to exclude residual streams.
- `AGGREGATION_POLICY` chooses the first assistant position (`"assistant"`) or the mean over all cached positions (`"all"`).

### Scatter-plot colors

Set `SCATTER_COLOR_METADATA_PATH` to a top-level prompt metadata key such as `"number_format"`, a dotted nested path such as `"task_metadata.difficulty"`, or an equivalent tuple such as `("task_metadata", "difficulty")`. Numeric values receive a continuous color scale and categorical values receive discrete colors. `None` retains log-time-horizon coloring.

In [ ]:
PROMPT_FRAMING: str | None = "task_available_time"
OUTPUT_FORMAT: str | None = None
TASK_METADATA_FILTERS: dict[str, object] | None = {
    "difficulty": "low",
    "domain": "communication",
}
SCATTER_COLOR_METADATA_PATH: str | tuple[str, ...] | None = None
COMPLETIONS_PATH = repo_root / "data" / "completions_256.jsonl"
COMPLETIONS_GCS_BUCKET = "temporal-research-bucket"
COMPLETIONS_GCS_PREFIX = "completions"
ACTIVATIONS_DIR = repo_root / "results" / "feature_geometry_after_assistant_residual_stream"
SELECTED_NODES_PATH = repo_root / "data" / "selected_nodes" / "final_500_eap_ig.pkl"
NODE_CLASSES: set[str] | None = {"p_generic", "n_generic"}
RESIDUAL_STREAM_LAYERS: set[int] | None = set()
AGGREGATION_POLICY = "assistant"  # assistant or all
MAX_FILES: int | None = 1000000
OVERWRITE = False

## 3. Select and download data

### 3.1 Download the completions index

Download `gs://temporal-research-bucket/completions/completions_256.jsonl` through the authenticated Google Cloud Storage API before filtering it. An existing local file is skipped unless `OVERWRITE` is `True`.

In [ ]:
download_activation_files(
    [COMPLETIONS_PATH],
    gcs_prefix=COMPLETIONS_GCS_PREFIX,
    overwrite=OVERWRITE,
    upload_root=COMPLETIONS_PATH.parent,
    bucket_name=COMPLETIONS_GCS_BUCKET,
)
print(f"Completions file: {COMPLETIONS_PATH}")

### 3.2 Filter completion records

Find activation paths whose completion metadata matches the configured filters, apply `MAX_FILES`, and preview the selected paths.

In [ ]:
matching_paths = find_activation_paths(
    prompt_framing=PROMPT_FRAMING,
    output_format=OUTPUT_FORMAT,
    task_metadata=TASK_METADATA_FILTERS,
    completions_path=COMPLETIONS_PATH,
    activations_dir=ACTIVATIONS_DIR,
)
paths_to_download = matching_paths if MAX_FILES is None else matching_paths[:MAX_FILES]

print(f"Found {len(matching_paths):,} matching activation files.")
print(f"Selected {len(paths_to_download):,} files for download.")
for path in paths_to_download[:10]:
    print(path)

### 3.3 Download activation caches and node definitions

Existing local files are skipped unless `OVERWRITE` is `True`. Activation files are downloaded from the hardcoded `conversational_after_assistant_residual_stream/results/feature_geometry_after_assistant_residual_stream` GCS path. The selected-node definitions use the repository-relative GCS convention.

In [ ]:
downloaded_paths = download_activation_files(
    paths_to_download,
    gcs_prefix=(
        "conversational_after_assistant_residual_stream/"
        "results/feature_geometry_after_assistant_residual_stream"
    ),
    overwrite=OVERWRITE,
    upload_root=ACTIVATIONS_DIR,
    bucket_name="temporal-research-bucket",
)
download_activation_files(
    [SELECTED_NODES_PATH],
    gcs_prefix="eap-ig/data/selected_nodes",
    upload_root=SELECTED_NODES_PATH.parent,
    bucket_name="temporal-research-bucket",
)

print(f"Download complete: {len(downloaded_paths):,} activation paths.")
print(f"Selected-node definitions: {SELECTED_NODES_PATH}")

## 4. Aggregate cached activations

Cached tensors begin as `batch x cached positions x features`; selected attention tensors additionally retain their head-feature dimension. The same policy is applied to every included MLP, attention, and complete residual-stream tensor:

- `assistant`: keep the first cached position.
- `all`: average every cached token position.

In [ ]:
if not downloaded_paths:
    raise ValueError("No activation files were selected.")

selected_node_groups = load_selected_node_groups(SELECTED_NODES_PATH)
print("Available node classes:", sorted(selected_node_groups))
allowed_nodes = nodes_for_classes(selected_node_groups, NODE_CLASSES)
print("Node-class filter:", "all" if NODE_CLASSES is None else sorted(NODE_CLASSES))
print(
    "Residual-stream layers:",
    "all" if RESIDUAL_STREAM_LAYERS is None else sorted(RESIDUAL_STREAM_LAYERS),
)

aggregated_by_file = []
for path in downloaded_paths:
    aggregated_by_file.append(
        aggregate_activation_file(
            path,
            AGGREGATION_POLICY,
            allowed_nodes,
            RESIDUAL_STREAM_LAYERS,
        )
    )

print(f"Aggregated {len(aggregated_by_file):,} files with policy={AGGREGATION_POLICY!r}.")
for activation_type, tensors in aggregated_by_file[0]["activations"].items():
    example_shapes = {name: tuple(tensor.shape) for name, tensor in list(tensors.items())[:3]}
    print(activation_type, example_shapes)
print("Retained selected nodes:", {
    name: len(indices)
    for name, indices in aggregated_by_file[0]["node_indices"].items()
})
print("Included residual streams:", aggregated_by_file[0]["included_residual_streams"])

## 5. Build the feature matrix and project with PCA

### 5.1 Flatten activations into one vector per sample

Concatenate the aggregated MLP vectors and flattened attention-head tensors in a stable iteration order, then stack the samples into a single matrix.

In [ ]:
import torch

feature_vectors = []
for sample in aggregated_by_file:
    sample_features = list(sample["activations"]["mlp"].values())
    sample_features.extend(
        attention.ravel()
        for attention in sample["activations"]["attn"].values()
    )
    feature_vectors.append(torch.cat(sample_features, dim=0))

activation_matrix = torch.stack(feature_vectors).to(torch.float)
print("Activation matrix shape:", tuple(activation_matrix.shape))

### 5.2 Compute the first three principal components

Fit a low-rank PCA basis and center the activation matrix before projecting each sample into three dimensions.

In [ ]:
_, _, principal_directions = torch.pca_lowrank(activation_matrix)
projection_directions = principal_directions[:, :3]
projs = (activation_matrix - activation_matrix.mean(dim=0, keepdim=True)) @ projection_directions
print("Projection shape:", tuple(projs.shape))

## 6. Visualize the PCA projections

Both plots use the same metadata extraction and color encoding. Missing paths and non-scalar metadata values raise explicit errors so configuration mistakes remain visible.

### 6.1 Static Matplotlib plot

Load prompt metadata for the selected samples, resolve the configured color field, and render a static scatter plot with either a numeric colorbar or a categorical legend.

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

# Prepare projections and the prompt metadata used to color them.
projs_np = projs.detach().cpu().numpy()

sample_indices = [
    int(path.stem.removeprefix("activations_sample_"))
    for path in downloaded_paths
]
target_indices = set(sample_indices)
completion_metadata = {}
with COMPLETIONS_PATH.open(encoding="utf-8") as completion_file:
    for index, line in enumerate(completion_file):
        if index in target_indices:
            completion_metadata[index] = json.loads(line)["prompt_metadata"]
            if len(completion_metadata) == len(target_indices):
                break

missing_indices = target_indices - completion_metadata.keys()
if missing_indices:
    raise ValueError(f"Missing completion metadata for sample indices: {sorted(missing_indices)}")


def get_metadata_value(metadata, path):
    """Resolve a dotted string or sequence of keys relative to prompt_metadata."""
    keys = path.split(".") if isinstance(path, str) else list(path)
    if not keys or any(not key for key in keys):
        raise ValueError("SCATTER_COLOR_METADATA_PATH must contain at least one non-empty key.")

    value = metadata
    for key in keys:
        if not isinstance(value, dict) or key not in value:
            raise KeyError(f"Metadata path {'.'.join(keys)!r} is missing at key {key!r}.")
        value = value[key]
    if isinstance(value, (dict, list, tuple, set)):
        raise TypeError(f"Metadata path {'.'.join(keys)!r} must resolve to a scalar value.")
    return value


if SCATTER_COLOR_METADATA_PATH is None:
    # Preserve the original canonical time-horizon coloring by default.
    seconds_to_months = 3.80517e-7
    unit_to_months = {
        "second": seconds_to_months,
        "seconds": seconds_to_months,
        "minute": 60 * seconds_to_months,
        "minutes": 60 * seconds_to_months,
        "hour": 60 * 60 * seconds_to_months,
        "hours": 60 * 60 * seconds_to_months,
        "day": 24 * 60 * 60 * seconds_to_months,
        "days": 24 * 60 * 60 * seconds_to_months,
        "week": 7 * 24 * 60 * 60 * seconds_to_months,
        "weeks": 7 * 24 * 60 * 60 * seconds_to_months,
        "month": 1.0,
        "months": 1.0,
        "year": 12.0,
        "years": 12.0,
        "decade": 120.0,
        "decades": 120.0,
        "century": 1_200.0,
        "centuries": 1_200.0,
        "millennium": 12_000.0,
        "millennia": 12_000.0,
    }
    time_horizon_months = []
    for index in sample_indices:
        metadata = completion_metadata[index]
        value = float(metadata.get("base_value", metadata["value"]))
        unit = metadata.get("base_unit", metadata["unit"])
        time_horizon_months.append(value * unit_to_months[unit.lower()])

    time_horizon_months = np.asarray(time_horizon_months)
    if np.any(time_horizon_months <= 0):
        raise ValueError("Time horizons must be positive before taking the logarithm.")
    plot_color_values = np.log10(time_horizon_months)
    plot_color_label = "log10(time horizon in months)"
    plot_color_is_numeric = True
else:
    metadata_path = (
        SCATTER_COLOR_METADATA_PATH.split(".")
        if isinstance(SCATTER_COLOR_METADATA_PATH, str)
        else list(SCATTER_COLOR_METADATA_PATH)
    )
    raw_color_values = [
        get_metadata_value(completion_metadata[index], metadata_path)
        for index in sample_indices
    ]
    plot_color_label = ".".join(metadata_path)
    plot_color_is_numeric = all(
        isinstance(value, (int, float, np.number))
        and not isinstance(value, (bool, np.bool_))
        for value in raw_color_values
    )
    plot_color_values = np.asarray(
        raw_color_values if plot_color_is_numeric else [str(value) for value in raw_color_values],
        dtype=float if plot_color_is_numeric else object,
    )

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")

if plot_color_is_numeric:
    scatter = ax.scatter(
        projs_np[:, 0], projs_np[:, 1], np.zeros(len(projs_np)),
        c=plot_color_values, cmap="viridis", marker="o", alpha=0.6,
    )
    fig.colorbar(scatter, ax=ax, pad=0.1, label=plot_color_label)
else:
    categories = list(dict.fromkeys(plot_color_values))
    category_to_index = {category: index for index, category in enumerate(categories)}
    categorical_cmap = plt.get_cmap("tab20", len(categories))
    point_colors = [categorical_cmap(category_to_index[value]) for value in plot_color_values]
    ax.scatter(
        projs_np[:, 0], projs_np[:, 1], np.zeros(len(projs_np)),
        c=point_colors, marker="o", alpha=0.6,
    )
    legend_handles = [
        Line2D([0], [0], marker="o", linestyle="", label=category,
               markerfacecolor=categorical_cmap(index), markeredgecolor="none")
        for index, category in enumerate(categories)
    ]
    ax.legend(handles=legend_handles, title=plot_color_label, bbox_to_anchor=(1.05, 1))

ax.set_xlabel("Component 1")
ax.set_ylabel("Component 2")
ax.set_zlabel("Component 3")
ax.set_title(f"3D Scatter Plot Colored by {plot_color_label}")

plt.show()

### 6.2 Interactive Plotly plot

Place the projections and shared color values into a DataFrame, then render an interactive plot with hover details.

In [ ]:
import plotly.express as px
import pandas as pd

df_projs = pd.DataFrame(projs_np, columns=["PC1", "PC2", "PC3"])
df_projs["color_value"] = plot_color_values
if SCATTER_COLOR_METADATA_PATH is None:
    df_projs["time_horizon_months"] = time_horizon_months

hover_data = (
    {"time_horizon_months": ":.6g"}
    if SCATTER_COLOR_METADATA_PATH is None
    else None
)

fig = px.scatter_3d(
    df_projs,
    x="PC1",
    y="PC2",
    z="PC3",
    color="color_value",
    color_continuous_scale="Viridis" if plot_color_is_numeric else None,
    hover_data=hover_data,
    labels={"color_value": plot_color_label},
    title=f"Interactive 3D Scatter Plot Colored by {plot_color_label}",
    opacity=0.7,
)

fig.update_traces(marker=dict(size=4))
fig.show()